# Lekcija 10 - AI agenti u produkciji

U ovoj lekciji naučit ćete **produkcijske obrasce** za AI agente koristeći Microsoft Agent Framework (Python). Obuhvaćamo:

- **Observabilnost** — dodavanje mjerenja vremena i evidentiranja interakcija agenta
- **Evaluacija** — korištenje evaluacijskog agenta za ocjenjivanje kvalitete odgovora
- **Upravljanje troškovima** — strategije za optimizaciju tokena i odabir modela

Scenarij je **putnički agent** koji pomaže korisnicima planirati putovanja, s uključenim nadzorom i evaluacijom.


## Postavljanje


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
import time
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())


## Razmatranja za produkciju

Premještanje AI agenata iz prototipova u produkciju zahtijeva pažljivu pozornost triju stupova:

1. **Promatljivost** — Potrebna vam je vidljivost u to što agent radi, koliko mu to traje i koje alate poziva. Bez praćenja i zapisivanja, otklanjanje pogrešaka u produkciji gotovo je nemoguće.

2. **Procjena** — Automatizirane provjere kvalitete osiguravaju da odgovori agenta ostanu točni, potpuni i korisni tijekom vremena. Agent procjenitelj može ocjenjivati odgovore prema definiranim kriterijima.

3. **Upravljanje troškovima** — Korištenje tokena izravno utječe na troškove. Strategije poput optimizacije upita, odabira modela i predmemoriranja pomažu držati troškove pod kontrolom bez žrtvovanja kvalitete.


## Izgradnja nadgledljivog agenta

Definiramo alate za putovanje i omotamo poziv agenta mjerenjem vremena kako bismo mogli pratiti kašnjenje. U produkciji biste integrirali OpenTelemetry ili sličan backend za praćenje.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## Uzorci evaluacije

Uobičajeni produkcijski obrazac je koristiti drugog agenta kao **procjenitelja**. Procjenitelj ocjenjuje odgovor primarnog agenta prema unaprijed definiranim kriterijima kao što su potpunost, točnost i korisnost.

To omogućuje:
- Automatizirane provjere kvalitete prije nego što odgovori stignu korisnicima
- Otkrivanje regresije kada se promijene upiti ili modeli
- Kontinuirano praćenje izvedbe agenta tijekom vremena


In [ ]:
evaluator = await provider.create_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## Strategije upravljanja troškovima

Kontrola troškova ključna je za produkcijske AI agente. Evo ključnih strategija:

| Strategy | Description |
|---|---|
| **Optimizacija prompta** | Držite sistemske upute sažetima. Uklonite redundantni kontekst kako biste smanjili broj ulaznih tokena. |
| **Odabir modela** | Koristite manje, jeftinije modele (npr. GPT-4o-mini) za jednostavne zadatke poput klasifikacije ili ekstrakcije, a veće modele ostavite za složeno zaključivanje. |
| **Predmemoriranje** | Predmemorirajte rezultate alata i česte upite kako biste izbjegli ponovljene API pozive. |
| **Ograničenja tokena** | Postavite ograničenja `max_tokens` kako biste spriječili neočekivano dugačke odgovore. |
| **Grupiranje** | Grupirajte više korisničkih upita u jedan API poziv gdje je moguće. |

U praksi, višeslojni pristup dobro funkcionira: usmjerite jednostavne zahtjeve na brz i jeftin model i eskalirajte samo složene upite na sposobniji model.


## Sažetak

U ovoj lekciji naučili ste kako:

1. **Dodati observability** u interakcije agenata pomoću mjerenja vremena i zapisivanja (logging), postavljajući temelje za praćenje (tracing) i nadzor.
2. **Automatski ocijeniti odgovore agenta** koristeći evaluacijski agent koji ocjenjuje cjelovitost, točnost i korisnost.
3. **Upravljati troškovima** kroz optimizaciju prompta, odabir modela, keširanje i budžete tokena.

Ovi obrasci za produkciju pomažu osigurati da su vaši AI agenti pouzdani, mjerljivi i isplativi na velikoj skali.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Odricanje odgovornosti**:
Ovaj dokument je preveden pomoću AI usluge za prevođenje [Co-op Translator](https://github.com/Azure/co-op-translator). Iako nastojimo postići točnost, imajte na umu da automatski prijevodi mogu sadržavati pogreške ili netočnosti. Izvorni dokument na izvornom jeziku smatra se mjerodavnim izvorom. Za kritične informacije preporučuje se profesionalni ljudski prijevod. Ne odgovaramo za bilo kakve nesporazume ili pogrešna tumačenja koja proizlaze iz korištenja ovog prijevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
